使用 spaCy 进行 NER

In [1]:
import spacy

# 加载英文模型
nlp = spacy.load("en_core_web_sm")

# 处理文本
text = "Apple is looking at buying U.K. startup for $1 billion"
doc = nlp(text)

# 输出识别结果
for ent in doc.ents:
    print(ent.text, ent.label_)

Apple ORG
U.K. GPE
$1 billion MONEY


基于规则的简单 NER 实现

In [2]:
import re

def rule_based_ner(text):
    # 匹配日期（格式：MM/DD/YYYY 或 MM-DD-YYYY）
    dates = re.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', text)
    # 匹配货币（格式：$ 后跟数字）
    currencies = re.findall(r'\$\d+\.?\d*', text)
    return {"日期": dates, "货币": currencies}

# 测试
sample = "会议定于12/15/2023举行，预算为$5000"
print(rule_based_ner(sample))

{'日期': ['12/15/2023'], '货币': ['$5000']}


补充 1：spaCy 支持的所有实体类型

In [3]:
import spacy

# 加载模型
nlp = spacy.load("en_core_web_sm")

# 获取所有实体标签及其描述
labels = nlp.get_pipe('ner').labels
for label in labels:
    print(label)

CARDINAL
DATE
EVENT
FAC
GPE
LANGUAGE
LAW
LOC
MONEY
NORP
ORDINAL
ORG
PERCENT
PERSON
PRODUCT
QUANTITY
TIME
WORK_OF_ART


补充 2：更完整的 spaCy NER 示例

In [4]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = "Elon Musk founded SpaceX in 2002 and Tesla in 2003."
doc = nlp(text)

for ent in doc.ents:
    print(f"实体: {ent.text}")
    print(f"  类型: {ent.label_}")
    print(f"  起始位置: {ent.start_char}")
    print(f"  结束位置: {ent.end_char}")
    print()

实体: Elon Musk
  类型: PERSON
  起始位置: 0
  结束位置: 9

实体: 2002
  类型: DATE
  起始位置: 28
  结束位置: 32

实体: Tesla
  类型: ORG
  起始位置: 37
  结束位置: 42

实体: 2003
  类型: DATE
  起始位置: 46
  结束位置: 50



补充 3：使用 NLTK 进行 NER

In [1]:
import nltk

# 设置数据搜索路径为当前项目下的 nltk_data 文件夹
nltk.data.path.append(r"./nltk_data")  # 或者写绝对路径 D:/11/NLP/nltk_data

# 此时就不需要再运行 download 了，直接使用
from nltk import pos_tag, ne_chunk
from nltk.tokenize import word_tokenize

text = "Elon Musk founded SpaceX in 2002."
tokens = word_tokenize(text)
pos_tags = pos_tag(tokens)
ner_tree = ne_chunk(pos_tags)

print(ner_tree)

(S
  (PERSON Elon/NNP)
  (PERSON Musk/NNP)
  founded/VBD
  (ORGANIZATION SpaceX/NNP)
  in/IN
  2002/CD
  ./.)


补充 4：使用 Stanford NER（通过 NLTK）

In [3]:
import os
from nltk.tag import StanfordNERTagger


# ==================================================
# 1. 配置Stanford NER文件路径
# ==================================================

stanford_ner_dir = r"D:\11\NLP\data\stanford-ner\stanford-ner-2020-11-17"

jar_path = os.path.join(
    stanford_ner_dir,
    "stanford-ner.jar"
)

model_path = os.path.join(
    stanford_ner_dir,
    "classifiers",
    "english.all.3class.distsim.crf.ser.gz"
)


# ==================================================
# 2. 检查文件
# ==================================================

if not os.path.isfile(jar_path):
    raise FileNotFoundError(
        f"没有找到JAR文件：\n{jar_path}"
    )

if not os.path.isfile(model_path):
    raise FileNotFoundError(
        f"没有找到模型文件：\n{model_path}"
    )

print("JAR文件检查成功")
print("模型文件检查成功")


# ==================================================
# 3. 创建Stanford NER识别器
# ==================================================

st = StanfordNERTagger(
    model_filename=model_path,
    path_to_jar=jar_path,
    encoding="utf-8",
    java_options="-mx1000m"
)


# ==================================================
# 4. 准备测试文本
# ==================================================

text = "Elon Musk founded SpaceX in 2002."

# StanfordNERTagger接收的是单词列表
tokens = text.replace(".", " .").split()

print("\n分词结果：")
print(tokens)


# ==================================================
# 5. 执行命名实体识别
# ==================================================

result = st.tag(tokens)

print("\n命名实体识别结果：")

for word, entity_type in result:
    print(f"{word:<12} {entity_type}")

JAR文件检查成功
模型文件检查成功

分词结果：
['Elon', 'Musk', 'founded', 'SpaceX', 'in', '2002', '.']

命名实体识别结果：
Elon         PERSON
Musk         PERSON
founded      O
SpaceX       ORGANIZATION
in           O
2002         O
.            O


补充 5：使用 Hugging Face Transformers 进行 NER

In [2]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

# ============================================================
# 1. 指向你刚才存放下载文件的本地文件夹
# ============================================================
model_path = r"D:\11\NLP\data\bert-base-ner-local"

# ============================================================
# 2. 从本地加载（加 local_files_only=True 强制离线）
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)
model = AutoModelForTokenClassification.from_pretrained(
    model_path,
    local_files_only=True
)

# ============================================================
# 3. 创建 NER pipeline
# ============================================================
ner_pipeline = pipeline("ner",model=model,tokenizer=tokenizer)

# ============================================================
# 4. 测试
# ============================================================
text = "Elon Musk founded SpaceX in 2002 and Tesla in 2003."
results = ner_pipeline(text)

for result in results:
    print(f"实体：{result['word']}，类型：{result['entity']}，置信度：{result['score']:.4f}")

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at D:\11\NLP\data\bert-base-ner-local were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


实体：El，类型：B-PER，置信度：0.5563
实体：##on，类型：I-ORG，置信度：0.5188
实体：Mu，类型：I-PER，置信度：0.7687
实体：##sk，类型：I-PER，置信度：0.5782
实体：Space，类型：B-ORG，置信度：0.9994
实体：##X，类型：I-ORG，置信度：0.9992
实体：Te，类型：B-ORG，置信度：0.9975
实体：##sla，类型：I-ORG，置信度：0.9670


补充 6：中文 NER（使用 spaCy 中文模型）

In [1]:
import spacy

# 加载中文模型
nlp_zh = spacy.load("zh_core_web_sm")

# 中文 NER 示例
text_zh = "马云是阿里巴巴的创始人，出生于1964年。"
doc_zh = nlp_zh(text_zh)

for ent in doc_zh.ents:
    print(f"实体: {ent.text}, 类型: {ent.label_}")

实体: 马云, 类型: PERSON
实体: 阿里巴巴, 类型: ORG
实体: 1964年, 类型: DATE
